# 06 · ADHD graph analysis (synchrony graphs of the attended windows)

**Purpose.** Check whether the windows MIL-CREDA's attention favours carry the class information, with a
classifier that is independent of MIL-CREDA. For each of the 121 target-test children of the ten ADHD units, the
broadband phase-locking value (PLV) between the 19 electrodes is computed per EEG window; a subject's graph is the
mean PLV over a set of its windows (all of them, the 25 % with the highest or lowest attention, or 25 % at random),
and XGBoost is trained on 39 graph descriptors (degree and eigenvector centrality per electrode, spectral radius).

**Paper items produced here**

* Figure `fig:ieee-gsa` (`ieee-gsa.pdf`): the 20 electrode pairs whose synchrony differs most between ADHD and
  control children, with all windows and with the highest-attention quarter, and the XGBoost balanced accuracy (BA).
* Figure `fig:ieee-pair-descriptors` (`ieee-pair-descriptors.pdf`): eigenvector centrality per electrode of the
  ADHD pair of `fig:ieee-alignment` (unit `ieee-r3h1`, target `IEEE-v236`, source `IEEE-v265`).
* The numbers of the paper's ADHD graph analysis paragraph (BA 0.645 -> 0.727, bootstrap interval, Holm p-values,
  F4-Cz rank-biserial correlations) and of the pair paragraph (descriptor correlation 0.988, ...).

**Where the data comes from**

* EEG windows: the public Kaggle dataset [`daprosero/mi-tdah-dataset`](https://www.kaggle.com/datasets/daprosero/mi-tdah-dataset)
  (CC0), read with `repro.ieee_root()` (mounted on Kaggle, downloaded with `kagglehub` elsewhere, about 33 MB).
* Attention per window: the MIL-CREDA cef pick of each unit in the ADHD pool (stage 04,
  `04_ADHD_Pool.ipynb`), condensed in `Results/paper/06_adhd_graph_analysis/summary/pick_attention.npz`. Nothing is
  retrained here: the attention is the one the pool saved.
* Unit configurations: the tracked phase-1 bases `MIL-CREDA-EEG/Manifests/ieee_pool_bases/`.

In [ ]:
# Setup: make `src/` importable, locally (notebook inside the repository) or on Kaggle (clone the repository).
import os, sys, subprocess
from pathlib import Path

REPO_URL = "https://github.com/Daprosero/Domain_Adaptation"
here = Path.cwd().resolve()
candidates = [here, *here.parents]
root = next((p for p in candidates if (p / "src" / "MIL_CREDA_EEG").is_dir()), None)
if root is None:                       # Kaggle / Colab: clone the public repository next to the notebook
    root = Path("/kaggle/working/Domain_Adaptation") if Path("/kaggle/working").exists() else here / "Domain_Adaptation"
    if not root.exists():
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, str(root)], check=True)
sys.path.insert(0, str(root / "src"))
print("repository:", root)

## Parameters

All values are the paper campaign's (protocol v84 sec 8.2/8.4), taken from its scripts `Results/v84-gsa/gsa84.py`
and `xgb84.py` and the figure scripts `fig_gsa84.py` / `fig_pair84.py`; the module ports them verbatim.

| parameter | value | source |
|---|---|---|
| units | 10 (`ieee-r0h0` ... `ieee-r4h1`) | `ieee_pool.UNITS` |
| attention | MIL-CREDA's cef pick per unit (trial, BA, AUC) | `Manifests/attention_influence/ieee_cef_picks.json` |
| PLV | common average, broadband Hilbert phase, zero diagonal | `gsa84.py` |
| window sets | all; top / bottom 25 % attention; random 25 % (10 draws, seeds 1000-1009) | `xgb84.py` |
| XGBoost | logistic, 200 trees, depth 3, rate 0.05, subsample 0.8, colsample 0.8 | `xgb84.py` (IWINAC 2026) |
| descriptor selection | top 6 by gain inside each training fold | `xgb84.py` |
| validation | stratified 5-fold x 10 repeats | `xgb84.py` |
| contrasts | paired bootstrap (10 000, stratified), Holm over 3 | `xgb84.py` |

**Cost of the full run.** The paper ran this stage on a local workstation, not on Kaggle (about 12 s per unit for
the PLV). Measured on a laptop CPU (Apple M-series, 4 XGBoost threads): PLV of the ten units about 90 s, GSA-XGB
about 3 min, so about 5 min in all, CPU only, no GPU. One Kaggle CPU session runs it whole; no split is needed,
but `SPLIT` works as in every stage.

In [ ]:
import json
from MIL_CREDA_EEG import repro
from MIL_CREDA_EEG.reproduce import stage06_gsa as S

SPLIT = None  # or (k, n) to run portion k of n in one Kaggle account
SMOKE = os.environ.get("REPRO_SMOKE") == "1"   # one unit and a reduced XGBoost, only to prove the wiring

STAGE_DIR = repro.results_dir(S.STAGE)
if SMOKE:                               # never overwrite the stored summary with a smoke result
    import tempfile
    WORK = Path(tempfile.mkdtemp(prefix="stage06-smoke-"))
    RUNS, SUMMARY = WORK / "runs", WORK / "summary"
else:
    RUNS, SUMMARY = STAGE_DIR / "runs", STAGE_DIR / "summary"
print("units in this portion:", repro.portion(S.UNITS, SPLIT))
print(json.dumps({u: S.picks()[u] for u in repro.portion(S.UNITS, SPLIT)}, indent=1)[:600], "...")

## Run: PLV per window

For each unit of the portion: rebuild its target-test and source subjects from the public data, compute the PLV of
every window, attach the stored attention of MIL-CREDA's pick, and check that the pick's saved test prediction
reproduces its BA and AUC (`reproduces: True`). Writes `runs/gsa84-<unit>.npz` (about 3.5 MB each; not tracked).

In [ ]:
IEEE_ROOT = repro.ieee_root()
reports = S.run(split=SPLIT, smoke=SMOKE, out=RUNS, ieee_root=IEEE_ROOT)

## Merge

When several accounts ran different portions, copy every account's `runs/` files into this stage's
`Results/paper/06_adhd_graph_analysis/runs/` folder first; `merge` then needs all ten units. It builds each
subject's graph per window set (`graphs.npz`), the attention-weighted graphs of the pair unit (`pair_graphs.npz`),
and runs the GSA-XGB analysis (`xgb84.json`, about 3 min). In smoke mode it accepts the one unit and runs a reduced
XGBoost into a temporary folder.

In [ ]:
out = S.merge(out=SUMMARY, runs=RUNS, smoke=SMOKE)
xg = json.loads((out / "xgb84.json").read_text())
print({k: round(v, 3) for k, v in xg["BA"].items()})

## Render the paper's figures and tables

From the stored summary only (no training): `figures/ieee-gsa.{pdf,png}`, `figures/ieee-pair-descriptors.{pdf,png}`,
`tables/gsa_xgb.{tex,csv}` and `tables/pair_descriptors.{tex,csv}`. In a full run `SUMMARY` is the stored summary
you just regenerated; in smoke mode the figures are drawn from the tracked summary.

In [ ]:
from IPython.display import Image, display
done = S.render(summary_dir=None if SMOKE else SUMMARY)
for name in ("ieee-gsa", "ieee-pair-descriptors"):
    display(Image(filename=str(STAGE_DIR / "figures" / f"{name}.png")))
print((STAGE_DIR / "tables" / "gsa_xgb.csv").read_text())
print((STAGE_DIR / "tables" / "pair_descriptors.csv").read_text())

## Compare with the paper

Each row quotes a number of `paper/main.tex` (line given) next to the one obtained here, rounded as the paper
prints it. From the stored summary every row matches exactly; a fresh run reproduced the stored graphs to 3e-9 and
the XGBoost result bit for bit on the machine where it was checked, but other library versions may move the last
digit.

In [ ]:
rows = repro.compare(S.paper_checks(None if SMOKE else SUMMARY))